# MV-IA-FraudGT — AML-Small-HI — one-seed check
This notebook runs the **full FraudGT encoder** (RMP + Ports + Ego IDs + FraudGT blocks) with the proposed multi-view gated head and class-weighted focal loss. Start with one seed; do not report it as a final multi-seed result.

In [ ]:
import platform, sys, subprocess, torch
print('Python:', sys.version)
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}; VRAM={p.total_memory / 1024**3:.2f} GiB')
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
import subprocess, sys, torch
torch_version = torch.__version__.split('+')[0]
cuda_tag = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'
print('PyG wheel index:', wheel_url)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyg_lib', 'torch_scatter', 'torch_sparse', '-f', wheel_url], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch_geometric', 'torchmetrics', 'yacs', 'datatable',
                'pandas', 'wandb', 'ogb', 'tensorboardX'], check=True)
print('Dependencies installed. Restart the session only if imports fail.')

In [ ]:
from pathlib import Path
import os, subprocess

REPO_URL = 'https://github.com/YOUR_USERNAME/MV-IA-FraudGT.git'  # CHANGE THIS
if 'YOUR_USERNAME' in REPO_URL:
    raise ValueError('Replace REPO_URL with your GitHub repository URL first.')
repo = Path('/kaggle/working/MV-IA-FraudGT')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
os.chdir(repo)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
from pathlib import Path
from shutil import copy2
candidates = list(Path('/kaggle/input').rglob('HI-Small_Trans.csv'))
if not candidates:
    raise FileNotFoundError('Add the IBM AML dataset as Kaggle Input; HI-Small_Trans.csv was not found.')
destination = Path('/kaggle/working/MV-IA-FraudGT/data/AML/HI-Small_Trans.csv')
destination.parent.mkdir(parents=True, exist_ok=True)
if not destination.exists() or destination.stat().st_size != candidates[0].stat().st_size:
    copy2(candidates[0], destination)
print('Dataset:', destination, f'({destination.stat().st_size / 1024**2:.1f} MiB)')

## One-seed training
The first run may spend roughly 10–20 minutes generating Ports on CPU before GPU training begins. The T4 configuration uses batch 256, fanout `[15,15]`, 128 iterations/epoch and 50 epochs.

In [ ]:
import os, subprocess, sys, time
os.chdir('/kaggle/working/MV-IA-FraudGT')
cmd = [sys.executable, '-u', '-m', 'fraudGT.main',
       '--cfg', 'configs/AML-Small-HI/AML-Small-HI-MV-IA-FraudGT-T4.yaml',
       '--repeat', '1', '--gpu', '0', 'name_tag', 'MVIA1Seed']
print('Command:', ' '.join(cmd))
started = time.time()
subprocess.run(cmd, check=True)
print(f'Elapsed: {(time.time() - started) / 60:.1f} minutes')

In [ ]:
from pathlib import Path
result_dirs = [p for p in Path('results').iterdir() if p.is_dir() and 'MVIA1Seed' in p.name]
print('Candidate result directories:')
for path in result_dirs:
    print(' -', path.resolve())
print('Use scripts/summarize_thresholds.py on the directory containing the seed subdirectory.')